# Tugas Software Testing — Kuliah Asinkron

| | |
|---|---|
| **Nama** | *(isi)* |
| **NIM** | *(isi)* |

**Cara mengerjakan.** Instruksi lengkap ada di dokumen *Kuliah Asinkron: Software Testing*. Ringkasnya:

* **Bagian A** – kerjakan Latihan **2, 3, 5, 6, dan 7** di bawah ini (Latihan 1, 4, dan 8 dianjurkan, tidak dinilai).
* **Bagian B** – mini-proyek `nilai_huruf` di akhir notebook.
* **Bagian C** – refleksi singkat di sel paling akhir.

Simpan notebook ini dengan nama `Tugas_Testing_<NIM>.ipynb`, jalankan **Restart & Run All**, lalu kumpulkan **beserta output-nya**. Bagian latihan memakai Bahasa Inggris agar sama dengan slide; jawaban boleh dalam Bahasa Indonesia.

# Software Testing — Hands-on Exercises (Python)

Companion notebook for the lecture **Software Testing** (adapted from I. Sommerville, *Engineering Software Products*, Ch. 9).

You will practise, in plain Python with the built-in `unittest` framework:

| # | Topic | Lecture slides |
|---|-------|----------------|
| 1 | Equivalence partitions | Unit testing, Fig. 9.3, Table 9.3 |
| 2 | Running and *reading* executable tests | Program 9.3 |
| 3 | Edge cases | Table 9.4 guidelines |
| 4 | Arrange – Action – Assert | Program 9.2 |
| 5 | Test-driven development | Fig. 9.8, Table 9.9 |
| 6 | Risk-based security testing | Table 9.11, risk analysis |
| 7 | Code review with a checklist | Table 9.13 |
| 8 | Running tests from files | Program 9.4 |

**How to use it.** Run the cells from top to bottom (Jupyter, VS Code or Google Colab — no extra packages needed). Cells marked **✏️ TODO** are for you. Every TODO cell runs without crashing even before you edit it; the test runner will simply report failures until your solution is right.

## 0. Setup

In [ ]:
import re
import unittest

def run_tests(*test_cases, verbosity=2):
    """Run one or more unittest.TestCase classes inside the notebook and print a summary."""
    suite = unittest.TestSuite()
    for tc in test_cases:
        suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(tc))
    result = unittest.TextTestRunner(verbosity=verbosity).run(suite)
    print(f"\n==> {result.testsRun} run, {len(result.failures)} failures, {len(result.errors)} errors")
    return result

The function under test is **Program 9.1** from the lecture, copied unchanged (only spacing is tidied):

In [ ]:
def namecheck(s):
    # Checks that a name only includes alphabetic characters, - or
    # a single quote. Names must be between 2 and 40 characters long
    # quoted strings and -- are disallowed
    namex = r"^[a-zA-Z][a-zA-Z-']{1,39}$"
    if re.match(namex, s):
        if re.search("'.*'", s) or re.search("--", s):
            return False
        else:
            return True
    else:
        return False

namecheck("O'Reilly"), namecheck("C-3PO")

## Exercise 1 — Equivalence partitions

Table 9.3 splits the inputs of `namecheck` into **2 correct** and **4 incorrect** partitions:

| Partition | Rule |
|---|---|
| `C1` Correct names 1 | only letters, 2–40 characters |
| `C2` Correct names 2 | letters, hyphens or apostrophes, 2–40 characters |
| `I1` Incorrect names 1 | 2–40 characters but disallowed characters |
| `I2` Incorrect names 2 | allowed characters but length 1 or > 40 |
| `I3` Incorrect names 3 | first character is a hyphen or an apostrophe |
| `I4` Incorrect names 4 | contains a double hyphen, quoted text, or both |

**✏️ TODO 1a.** For each input below, write the partition you think it belongs to *and* whether `namecheck` should return `True` or `False`. Then run the cell to compare your prediction with the real result.

In [ ]:
# input            : (your partition, your predicted result)
predictions = {
    "Kurniawan"     : ("??", None),
    "Mary-Jane"     : ("??", None),
    "R2D2"          : ("??", None),
    "'Connor"       : ("??", None),
    "X"             : ("??", None),
    "Smith--Jones"  : ("??", None),
}

for name, (partition, predicted) in predictions.items():
    actual = namecheck(name)
    mark = "✓" if predicted == actual else "✗"
    print(f"{mark} {name!r:16} partition={partition:3} predicted={predicted!s:5} actual={actual}")

**✏️ TODO 1b.** Find **one more input per partition** that you have not seen yet in the lecture and add it to `my_examples`. The cell checks that the result matches the partition type (correct → `True`, incorrect → `False`).

In [ ]:
my_examples = {
    "C1": "",   # e.g. a name with letters only
    "C2": "",
    "I1": "",
    "I2": "",
    "I3": "",
    "I4": "",
}

for part, name in my_examples.items():
    expected = part.startswith("C")
    status = "OK " if name and namecheck(name) == expected else "TODO"
    print(f"{status} {part}: {name!r}")

## Exercise 2 — Run (and read!) the executable tests of Program 9.3

Below are the 14 tests of Program 9.3, unchanged. Run them.

In [ ]:
class TestNameCheck(unittest.TestCase):

    def test_alphaname(self):
        self.assertTrue(namecheck('Sommerville'))

    def test_doublequote(self):
        self.assertFalse(namecheck("Thisis'maliciouscode'"))

    def test_namestartswithhyphen(self):
        self.assertFalse(namecheck('-Sommerville'))

    def test_namestartswithquote(self):
        self.assertFalse(namecheck("'Reilly"))

    def test_nametoolong(self):
        self.assertFalse(namecheck('Thisisalongstringwithmorethen40charactersfrombeginningtoend'))

    def test_nametooshort(self):
        self.assertFalse(namecheck('S'))

    def test_namewithdigit(self):
        self.assertFalse(namecheck('C-3PO'))

    def test_namewithdoublehyphen(self):
        self.assertFalse(namecheck('--badcode'))

    def test_namewithhyphen(self):
        self.assertTrue(namecheck('Washington-Wilson'))

    def test_namewithinvalidchar(self):
        self.assertFalse(namecheck('Sommer_ville'))

    def test_namewithquote(self):
        self.assertTrue(namecheck("O'Reilly"))

    def test_namewithspaces(self):
        self.assertFalse(namecheck('Washington Wilson'))

    def test_shortname(self):
        self.assertTrue('Sx')

    def test_thiswillfail(self):
        self.assertTrue(namecheck("O Reilly"))

run_tests(TestNameCheck);

**Questions**

1. Which test fails, and why is that *expected*?
2. One test **can never fail**, whatever `namecheck` does. Which one, and why? *(Hint: what is `bool('Sx')`?)*

**✏️ TODO 2.** Fix the test that can never fail in the class below, then prove your fix works: temporarily break `namecheck` (the `broken_namecheck` cell) and check that your fixed test now catches the break.

In [ ]:
class TestShortName(unittest.TestCase):
    def test_shortname(self):
        self.assertTrue('Sx')          # ✏️ TODO: fix this assertion

run_tests(TestShortName);

In [ ]:
# Mutation check: a version of namecheck that wrongly rejects 2-character names.
_original = namecheck

def broken_namecheck(s):
    return _original(s) and len(s) > 2

namecheck = broken_namecheck          # swap in the broken version
try:
    print("With a BROKEN namecheck, a good test_shortname should FAIL:")
    run_tests(TestShortName, verbosity=1)
finally:
    namecheck = _original             # always restore the real function

## Exercise 3 — Edge cases (Table 9.4 guidelines)

`namecheck` accepts names of **2 to 40** characters. The guideline *“test edge cases”* says to pick inputs **at** the boundaries, and *“don't forget null and zero”* says to try the empty string.

**✏️ TODO 3.** Complete the five tests. Use `'A' * n` to build a name of length `n`.

In [ ]:
class TestNameCheckEdges(unittest.TestCase):

    def test_length_1_rejected(self):
        pass   # ✏️ TODO

    def test_length_2_accepted(self):
        pass   # ✏️ TODO

    def test_length_40_accepted(self):
        pass   # ✏️ TODO

    def test_length_41_rejected(self):
        pass   # ✏️ TODO

    def test_empty_string_rejected(self):
        pass   # ✏️ TODO

run_tests(TestNameCheckEdges);

⚠️ An empty test body (`pass`) *passes*. A passing test only means something if it contains an assertion — the same lesson as Exercise 2.

**Discussion.** Run the next cell. Is `namecheck` right to reject these names? Is this a *programming* error or an *understanding* error (slide “Program bugs”)?

In [ ]:
for name in ["D'Arcy-O'Neil", "Zoë", "José"]:
    print(f"{name!r:16} -> {namecheck(name)}")

## Exercise 4 — Arrange, Action, Assert

Program 9.2 tests an `interest_calculator(p, r, n)` whose code is **not shown** in the lecture. Here we use our **own** simple definition so that you can practise the AAA pattern:

> `simple_interest(p, r, n)` = interest on principal `p` at an annual rate of `r` % for `n` days, using a 365-day year, rounded to 2 decimals.

*(This is not Sommerville's calculator — with this formula the inputs `p=17000, r=3, n=365` give 510.0, not the 270.36 expected in Program 9.2, so the book's function must use a different formula.)*

In [ ]:
def simple_interest(p, r, n):
    return round(p * (r / 100) * (n / 365), 2)

**✏️ TODO 4.** Write three tests, each with clearly marked `# Arrange`, `# Action`, `# Assert` sections:

1. zero principal → interest 0
2. zero days → interest 0
3. one full year: `p = 1000`, `r = 5`, `n = 365` → 50.0

Bonus: `assertEqual` on floats is fragile. Look up `assertAlmostEqual` and use it in a fourth test with `n = 30`.

In [ ]:
class TestSimpleInterest(unittest.TestCase):

    def test_zero_principal(self):
        # Arrange
        # Action
        # Assert
        pass   # ✏️ TODO

    def test_zero_days(self):
        pass   # ✏️ TODO

    def test_one_year(self):
        pass   # ✏️ TODO

run_tests(TestSimpleInterest);

## Exercise 5 — Test-driven development

Follow the TDD cycle from Table 9.9: **write a test → write a stub that fails → run all tests → implement → rerun → refactor.**

The goal is a function `average(values)` that:

* returns the arithmetic mean of a non-empty list of numbers;
* for a single value returns that value (*“one is different”*);
* raises `ValueError` for an empty list (*“don't forget null and zero”*).

**Step 1 – tests first.** These tests are written for you. Run them against the stub — they must fail (red).

In [ ]:
class TestAverage(unittest.TestCase):

    def test_several_values(self):
        self.assertEqual(average([2, 4, 6]), 4)

    def test_single_value(self):
        self.assertEqual(average([7]), 7)

    def test_empty_list_raises(self):
        with self.assertRaises(ValueError):
            average([])

def average(values):        # stub: deliberately incomplete
    pass

run_tests(TestAverage);

**✏️ TODO 5a (green).** Re-define `average` so that **all three** tests pass. Rerun the tests.

**✏️ TODO 5b (new mini-unit).** Add a test `test_negative_values` (e.g. `[-2, 2]` → `0`) *before* changing any code. Does it pass immediately? What does that tell you?

**✏️ TODO 5c (refactor).** If your code can be simpler, simplify it, then rerun all tests.

In [ ]:
def average(values):
    pass   # ✏️ TODO

run_tests(TestAverage);

## Exercise 6 — Risk-based security testing

Risk 1 in Table 9.11 (*attacker gains access using authorized credentials*) can arise because *the user has set a weak password*. The lecture suggests a test that checks that the password-setting code **always checks password strength**.

We define our **own** policy for this exercise (it is an illustration, not a standard):

1. at least **12** characters;
2. not in the list `COMMON_PASSWORDS` (case-insensitive);
3. contains characters from at least **3** of these 4 classes: lowercase, uppercase, digit, symbol.

The tests from the lecture slide, plus a few more, are ready. **✏️ TODO 6.** Implement `is_strong_password` so they all pass.

In [ ]:
COMMON_PASSWORDS = {"password", "123456", "qwerty", "letmein", "password123", "admin"}

def is_strong_password(pw):
    return True    # ✏️ TODO: current version accepts everything (insecure!)

class TestPasswordPolicy(unittest.TestCase):
    # from the lecture slide
    def test_common_password_rejected(self):
        self.assertFalse(is_strong_password('password'))

    def test_short_password_rejected(self):
        self.assertFalse(is_strong_password('Ab1!'))

    def test_long_mixed_password_accepted(self):
        self.assertTrue(is_strong_password('Tr0ub4dor&3-horse'))

    # extra tests: edges and partitions of our policy
    def test_exactly_12_chars_three_classes_accepted(self):
        self.assertTrue(is_strong_password('abcdefgh12AB'))

    def test_11_chars_rejected(self):
        self.assertFalse(is_strong_password('abcdefgh12A'))

    def test_long_but_one_class_rejected(self):
        self.assertFalse(is_strong_password('abcdefghijklmnop'))

    def test_common_password_case_insensitive(self):
        self.assertFalse(is_strong_password('PASSWORD123'))

    def test_empty_rejected(self):
        self.assertFalse(is_strong_password(''))

run_tests(TestPasswordPolicy);

## Exercise 7 — Code review with a checklist

You are the reviewer. Use the Python checklist from Table 9.13 on the function below:

* Are meaningful variable and function names used?
* Have all data errors been considered and tests written for them?
* Are all exceptions explicitly handled?
* Are default function parameters used?
* Are types used consistently?
* Is the indentation level correct?

**✏️ TODO 7a.** Write down at least **three** issues (double-click this cell to edit):

1. …
2. …
3. …

In [ ]:
def f(x, l=[]):
    # adds a student score (0-100) to a list and returns the list
    if x >= 0 and x <= 100:
        l.append(x)
    else:
        l.append("invalid")
    return l

**✏️ TODO 7b.** These tests describe the behaviour the reviewer and programmer agree on. Run them against `f` to see which issues they reveal, then write a fixed `add_score` that passes them all.

In [ ]:
def add_score(score, scores=None):
    return f(score, scores) if scores is not None else f(score)   # ✏️ TODO: replace with a fixed version

class TestAddScore(unittest.TestCase):

    def test_calls_do_not_share_state(self):
        first = add_score(50)
        second = add_score(60)
        self.assertEqual(second, [60])

    def test_appends_to_given_list(self):
        self.assertEqual(add_score(70, [10]), [10, 70])

    def test_out_of_range_raises_value_error(self):
        with self.assertRaises(ValueError):
            add_score(101, [])

    def test_non_number_raises_type_error(self):
        with self.assertRaises(TypeError):
            add_score("80", [])

    def test_boundaries_accepted(self):
        self.assertEqual(add_score(0, []), [0])
        self.assertEqual(add_score(100, []), [100])

run_tests(TestAddScore);

## Exercise 8 — Running tests from files (Program 9.4)

In a real project tests live in files such as `test_namecheck.py`, and a runner *discovers* them. The next cells write a small project into a folder and run Program 9.4 on it.

In [ ]:
import os, pathlib, subprocess, sys, textwrap

proj = pathlib.Path("testing_demo")
proj.mkdir(exist_ok=True)

(proj / "RE_checker.py").write_text(textwrap.dedent("""
    import re

    def namecheck(s):
        namex = r"^[a-zA-Z][a-zA-Z-']{1,39}$"
        if re.match(namex, s):
            if re.search("'.*'", s) or re.search("--", s):
                return False
            else:
                return True
        else:
            return False
"""))

(proj / "test_namecheck.py").write_text(textwrap.dedent("""
    import unittest
    from RE_checker import namecheck

    class TestNameCheck(unittest.TestCase):
        def test_alphaname(self):
            self.assertTrue(namecheck('Sommerville'))
        def test_namewithquote(self):
            self.assertTrue(namecheck("O'Reilly"))
        def test_nametooshort(self):
            self.assertFalse(namecheck('S'))
"""))

# Program 9.4, saved as a script
(proj / "run_tests.py").write_text(textwrap.dedent("""
    import unittest

    loader = unittest.TestLoader()
    # Find the test files in the current directory
    tests = loader.discover('.')
    # Specify the level of information provided by the test runner
    testRunner = unittest.runner.TextTestRunner(verbosity=2)
    testRunner.run(tests)
"""))

print(sorted(p.name for p in proj.iterdir()))

In [ ]:
out = subprocess.run([sys.executable, "run_tests.py"], cwd=proj, capture_output=True, text=True)
print(out.stderr)   # unittest writes its report to stderr

**✏️ TODO 8.** Add a second file `test_edges.py` to `testing_demo/` containing your edge-case tests from Exercise 3 (re-run the cell above afterwards). Discovery picks up every file whose name matches `test*.py` — confirm that the number of tests run goes up.

Command-line equivalent: `python -m unittest` (run inside the folder).

In [ ]:
# ✏️ TODO: write testing_demo/test_edges.py here, e.g. with (proj / "test_edges.py").write_text(...)

---
# Bagian B — Mini-proyek: fungsi `nilai_huruf`

Anda akan menerapkan partisi ekuivalen, edge case, pola Arrange–Action–Assert, dan TDD pada satu fungsi kecil.

### Spesifikasi (khusus untuk tugas ini, bukan aturan nilai resmi)

`nilai_huruf(skor)` menerima skor angka dan mengembalikan nilai huruf:

| Skor | Hasil |
|---|---|
| 80 ≤ skor ≤ 100 | `'A'` |
| 70 ≤ skor < 80 | `'B'` |
| 60 ≤ skor < 70 | `'C'` |
| 50 ≤ skor < 60 | `'D'` |
| 0 ≤ skor < 50 | `'E'` |
| skor < 0 atau skor > 100 | `ValueError` |
| bukan `int`/`float` (termasuk `bool`, `str`, `None`) | `TypeError` |

### B1. Partisi ekuivalen dan nilai batas

✏️ Lengkapi tabel berikut (klik dua kali sel ini untuk mengedit). Tambahkan baris jika perlu. Setelah itu tuliskan **minimal 12 nilai batas** yang akan Anda uji, beserta hasil yang diharapkan.

| Partisi | Valid / tidak valid | Contoh input | Hasil yang diharapkan |
|---|---|---|---|
| | | | |

**Nilai batas yang diuji:**

1. …

### B2. Kembangkan `nilai_huruf` dengan TDD

Lakukan **minimal 3 siklus TDD**. Untuk setiap siklus, buat sel-sel berikut secara berurutan dan **biarkan output-nya terlihat**:

1. Sel markdown: `Siklus n – <apa yang ditambahkan>`.
2. Sel kode: tes baru (tambahkan ke kelas tes) dan jalankan → hasil **merah** (gagal).
3. Sel kode: kode baru/perbaikan `nilai_huruf` dan jalankan ulang **semua** tes → hasil **hijau**.

Contoh pembagian siklus: (1) skor valid → huruf; (2) skor di luar 0–100 → `ValueError`; (3) tipe salah → `TypeError`. Siklus pertama sudah dimulai di bawah sebagai contoh format.

#### Siklus 1 – skor valid menghasilkan huruf yang benar

In [ ]:
class TestNilaiHuruf(unittest.TestCase):

    def test_skor_90_mendapat_A(self):
        # Arrange
        skor = 90
        # Action
        hasil = nilai_huruf(skor)
        # Assert
        self.assertEqual(hasil, 'A')

def nilai_huruf(skor):      # stub: sengaja belum lengkap
    pass

run_tests(TestNilaiHuruf);

In [ ]:
# ✏️ Lanjutkan siklus 1: tulis kode yang membuat tes di atas lulus, lalu jalankan semua tes.

### B3. Kumpulan tes lengkap

✏️ Pada akhir B2, kelas `TestNilaiHuruf` Anda harus berisi **minimal 15 tes**, masing-masing dengan komentar `# Arrange`, `# Action`, `# Assert`. Tes harus mencakup semua partisi dan semua nilai batas dari B1. Jalankan sel di bawah untuk memastikan semuanya lulus terhadap implementasi Anda.

In [ ]:
run_tests(TestNilaiHuruf);

### B4. Tes Anda melawan implementasi orang lain

Seorang rekan menulis `nilai_huruf_v2` di bawah ini. Implementasi ini **mengandung bug**. Jangan ubah kodenya.

✏️ Jalankan tes Anda terhadap `nilai_huruf_v2` memakai `uji_dengan`. Lalu, di sel markdown setelahnya, jelaskan **setiap** tes yang gagal: bug apa yang ditemukan, di baris mana, dan nilai batas atau partisi mana yang menemukannya. Jika tidak ada tes yang gagal, tambahkan tes sampai bug-nya tertangkap.

In [ ]:
def nilai_huruf_v2(skor):
    if isinstance(skor, bool) or not isinstance(skor, (int, float)):
        raise TypeError("skor harus berupa angka")
    if skor < 0 or skor > 101:
        raise ValueError("skor harus di antara 0 dan 100")
    if skor > 80:
        return 'A'
    elif skor >= 70:
        return 'B'
    elif skor >= 60:
        return 'C'
    elif skor >= 50:
        return 'D'
    else:
        return 'E'

def uji_dengan(fungsi, *kelas_tes):
    """Jalankan kelas tes dengan `nilai_huruf` sementara diganti `fungsi`."""
    g = globals()
    asli = g.get('nilai_huruf')
    g['nilai_huruf'] = fungsi
    try:
        return run_tests(*kelas_tes, verbosity=1)
    finally:
        g['nilai_huruf'] = asli

uji_dengan(nilai_huruf_v2, TestNilaiHuruf);

✏️ **Analisis bug** (klik dua kali untuk mengedit):

| Tes yang gagal | Bug yang ditemukan | Baris di `nilai_huruf_v2` | Partisi / nilai batas |
|---|---|---|---|
| | | | |

---
# Bagian C — Refleksi (sekitar 300 kata)

✏️ Jawab dalam satu atau dua paragraf:

1. Dari pengalaman Bagian B, apa satu kelebihan dan satu kesulitan TDD yang Anda rasakan sendiri? Bandingkan dengan Table 9.10.
2. Bagaimana Anda memastikan tes yang Anda tulis **bisa gagal** (tidak seperti `test_shortname`)?

*Jawaban Anda:*